config block

In [0]:
input_path   = "/Workspace/Users/parul.datta.official@gmail.com/emissions-databricks/Emissions_Data_2023.csv"
staging_path = "/Volumes/emission/default/emissions_volume/staging/emissions_clean"
output_path  = "/Volumes/emission/default/emissions_volume/output/emissions_dashboard"

Reading the csv file.

In [0]:
df = spark.read.csv(input_path, header=True, inferSchema=False)


In [0]:
display(df)

Renaming columns: 
(1) "GHG emissions mtons CO2e" to ghg_emissions_mtons (2) "consolidated_city-county" to "consolidated_city_county"

In [0]:
df = df.withColumnRenamed("GHG emissions mtons CO2e", "ghg_emissions_mtons") \
       .withColumnRenamed("consolidated_city-county", "consolidated_city_county")

In [0]:
display(df.select("ghg_emissions_mtons", "consolidated_city_county"))

(1) Removing commas and casting to numbers       
(2) converting null value in consolidated_city_county column to N 

In [0]:
from pyspark.sql.functions import col, regexp_replace, when, expr
from pyspark.sql.types import DoubleType

text_cols = {"state_abbr", "county_state_name", "county_name",
             "consolidated_city_county", "county_id", "state_id", "area (sq. ft.)"} 

numeric_cols = [c for c in df.columns if c not in text_cols]

for c in numeric_cols:
    cleaned = regexp_replace(col(c), ",", "")
    df = df.withColumn(c, expr(f"try_cast(regexp_replace(`{c}`, ',', '') as DOUBLE)"))


# (2)
df = df.withColumn(
    "consolidated_city_county",
    when(col("consolidated_city_county").isNull(), "N")
    .otherwise(col("consolidated_city_county"))
)

In [0]:
display(df.select("ghg_emissions_mtons", "population", "consolidated_city_county"))


Dropping fully empty columns

In [0]:
from pyspark.sql.functions import count

total_rows = df.count()
null_counts = df.select([
    count(when(expr(f"`{c}`").isNull(), 1)).alias(c) for c in df.columns
]).collect()[0].asDict()

cols_to_drop = [c for c, n in null_counts.items() if n == total_rows]
print(f"Dropping {len(cols_to_drop)} empty columns: {cols_to_drop}")
df = df.drop(*cols_to_drop)

(1) Replace spaces and special characters with underscores
(2) Remove consecutive underscores
(3) Strip leading/trailing underscores
(4) Rename all columns




In [0]:
import re

def clean_column_name(col_name):
    clean = re.sub(r'[\s,;{}()\n\t=./]', '_', col_name)
    clean = re.sub(r'_+', '_', clean)
    clean = clean.strip('_').lower()
    return clean

new_col_names = {c: clean_column_name(c) for c in df.columns}
for old, new in new_col_names.items():
    if old != new:
        df = df.withColumnRenamed(old, new)

print(df.columns)

Filtering bad rows : missing values in population or ghg_emission_mtons

In [0]:
df = df.filter(col("population").isNotNull() & col("ghg_emissions_mtons").isNotNull())

3142 shows that there are no missing values for every US county

In [0]:
print(df.count())

Pre-Calculating dashboard metric : emissions per person

In [0]:
from pyspark.sql.functions import round as spark_round

df = df.withColumn(
    "emissions_per_person",
    spark_round(col("ghg_emissions_mtons") / col("population"), 4)
)

In [0]:
display(df.select("county_state_name", "ghg_emissions_mtons", "population", "emissions_per_person"))


Staging Parquet file

In [0]:
df.write.mode("overwrite").option("compression", "snappy").parquet(staging_path)

row_count = df.count()
col_count = len(df.columns)

dbutils.jobs.taskValues.set(key="staging_path", value=staging_path)
dbutils.jobs.taskValues.set(key="row_count",    value=row_count)
dbutils.jobs.taskValues.set(key="col_count",    value=col_count)

print(f"Notebook 1 done — {row_count} rows, {col_count} columns written to staging")

In [0]:
#spark.sql("SHOW VOLUMES IN emission.default").show()

In [0]:
#spark.sql("CREATE VOLUME IF NOT EXISTS emission.default.emissions_volume")